# QAOA Hyperparameter Sweep -- VRP (4 Customers)

Sweep across circuit depth (p), penalty, optimizer (COBYLA/ADAM/SPSA), maxiter, restarts, and fleet size (k).

Three instances (depot + 4 customers each). Position-indexed QUBO for k=1, edge-based for k>1.

In [19]:
!pip install -q qiskit==1.4.2 qiskit-aer==0.17.0 qiskit-algorithms==0.3.1 qiskit-optimization==0.6.1

In [20]:
import numpy as np
import itertools, time, hashlib, csv, math, warnings
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict

from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Estimator
from qiskit_algorithms.optimizers import ADAM, COBYLA, SPSA
from qiskit_algorithms.utils import algorithm_globals as ag
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

warnings.filterwarnings("ignore", category=DeprecationWarning)

@dataclass
class Node:
    id: int; x: float; y: float; demand: float = 0.0; is_depot: bool = False

def build_dist_map(nodes, depot, dist_matrix):
    all_n = [depot] + nodes
    return {(a.id, b.id): float(dist_matrix[a.id][b.id]) for a in all_n for b in all_n}

def route_distance(route_ids, depot_id, dist_map):
    if not route_ids: return 0.0
    d = dist_map[(depot_id, route_ids[0])]
    for idx in range(len(route_ids)-1):
        d += dist_map[(route_ids[idx], route_ids[idx+1])]
    d += dist_map[(route_ids[-1], depot_id)]
    return d

def classical_optimal(node_ids, eff_k, depot_id, dist_map):
    if eff_k == 1:
        return min(route_distance(list(p), depot_id, dist_map)
                   for p in itertools.permutations(node_ids))
    def _parts(ids, k):
        ids_l = list(ids); n = len(ids_l)
        if k == 1: yield (tuple(ids_l),); return
        if n == k: yield tuple((x,) for x in ids_l); return
        if n < k or k <= 0: return
        first = ids_l[0]; rest = ids_l[1:]
        for sub in _parts(tuple(rest), k - 1):
            yield ((first,),) + sub
        for sub in _parts(tuple(rest), k):
            for gi in range(len(sub)):
                yield sub[:gi] + ((first,) + sub[gi],) + sub[gi+1:]
    best = float("inf")
    for part in _parts(tuple(node_ids), eff_k):
        if len(part) != eff_k: continue
        for combo in itertools.product(*[itertools.permutations(b) for b in part]):
            total = sum(route_distance(list(p), depot_id, dist_map) for p in combo)
            if total < best: best = total
    return best

print("Imports and helpers loaded.")


Imports and helpers loaded.


In [21]:
INSTANCES = {}

# -- RioClaroPostToy_5_0 (4 customers, node 5 dropped) --
_dm_i0 = np.array([
    [0.000, 3515.888, 1995.381, 3050.876, 2215.837],
    [3515.888, 0.000, 2582.476, 4167.265, 4164.734],
    [1995.381, 2582.476, 0.000, 1832.876, 1678.343],
    [3050.876, 4167.265, 1832.876, 0.000, 2067.479],
    [2215.837, 4164.734, 1678.343, 2067.479, 0.000]
])
_depot_i0 = Node(id=0, x=5215.0, y=5704.0, demand=0, is_depot=True)
_nodes_i0 = [
    Node(id=1, x=4728.0, y=2348.0, demand=1.0),
    Node(id=2, x=6287.0, y=4357.0, demand=1.0),
    Node(id=3, x=8111.0, y=4162.0, demand=1.0),
    Node(id=4, x=7097.0, y=5700.0, demand=1.0)
]
INSTANCES["RioClaroPostToy_5_0"] = {
    "dist_matrix": _dm_i0, "depot": _depot_i0, "nodes": _nodes_i0,
    "dist_map": build_dist_map(_nodes_i0, _depot_i0, _dm_i0),
    "global_max": float(np.max(_dm_i0)),
}

# -- RioClaroPostToy_5_1 (4 customers, node 5 dropped) --
_dm_i1 = np.array([
    [0.000, 3525.430, 3001.988, 2275.052, 5885.457],
    [3525.430, 0.000, 729.503, 1925.706, 7556.102],
    [3001.988, 729.503, 0.000, 1341.462, 7149.288],
    [2275.052, 1925.706, 1341.462, 0.000, 6018.115],
    [5885.457, 7556.102, 7149.288, 6018.115, 0.000]
])
_depot_i1 = Node(id=0, x=5215.0, y=5704.0, demand=0, is_depot=True)
_nodes_i1 = [
    Node(id=1, x=4495.0, y=2366.0, demand=1.0),
    Node(id=2, x=4767.0, y=2864.0, demand=1.0),
    Node(id=3, x=5483.0, y=3800.0, demand=1.0),
    Node(id=4, x=11214.0, y=5292.0, demand=1.0)
]
INSTANCES["RioClaroPostToy_5_1"] = {
    "dist_matrix": _dm_i1, "depot": _depot_i1, "nodes": _nodes_i1,
    "dist_map": build_dist_map(_nodes_i1, _depot_i1, _dm_i1),
    "global_max": float(np.max(_dm_i1)),
}

# -- RioClaroPostToy_5_2 (4 customers, node 5 dropped) --
_dm_i2 = np.array([
    [0.000, 3553.496, 2744.139, 1931.003, 2838.503],
    [3553.496, 0.000, 3449.807, 2087.657, 1586.824],
    [2744.139, 3449.807, 0.000, 2948.947, 1866.527],
    [1931.003, 2087.657, 2948.947, 0.000, 2104.400],
    [2838.503, 1586.824, 1866.527, 2104.400, 0.000]
])
_depot_i2 = Node(id=0, x=5215.0, y=5704.0, demand=0, is_depot=True)
_nodes_i2 = [
    Node(id=1, x=4571.0, y=2360.0, demand=1.0),
    Node(id=2, x=7313.0, y=4083.0, demand=1.0),
    Node(id=3, x=4227.0, y=4332.0, demand=1.0),
    Node(id=4, x=5917.0, y=3341.0, demand=1.0)
]
INSTANCES["RioClaroPostToy_5_2"] = {
    "dist_matrix": _dm_i2, "depot": _depot_i2, "nodes": _nodes_i2,
    "dist_map": build_dist_map(_nodes_i2, _depot_i2, _dm_i2),
    "global_max": float(np.max(_dm_i2)),
}

for iname, idata in INSTANCES.items():
    print(f"{iname}: {len(idata['nodes'])} customers, max_dist={idata['global_max']:.1f}")

RioClaroPostToy_5_0: 4 customers, max_dist=4167.3
RioClaroPostToy_5_1: 4 customers, max_dist=7556.1
RioClaroPostToy_5_2: 4 customers, max_dist=3553.5


In [ ]:
SWEEP_REPS      = [1, 2, 3, 5]
SWEEP_PENALTY   = [2.0, 5.0, 10.0]
SWEEP_OPTIMIZER = ["COBYLA", "ADAM","SPSA"]
SWEEP_MAXITER   = [50, 200]           # maxiter doesn't matter for COBYLA/ADAM; 2 values to prove it
SWEEP_RESTARTS  = [1, 3]              # r=5 is marginal over r=3
SWEEP_K         = [1, 2, 3, 4]
SHOTS           = 10_000              # 10K is enough to measure valid fractions

SEED     = 7
DECODE_K = 10

n_cfg = (len(SWEEP_REPS) * len(SWEEP_PENALTY) * len(SWEEP_OPTIMIZER)
         * len(SWEEP_MAXITER) * len(SWEEP_RESTARTS) * len(SWEEP_K))
print(f"Grid: {n_cfg} configs/instance x {len(INSTANCES)} instances = {n_cfg * len(INSTANCES)} total runs")


Grid: 384 configs/instance x 3 instances = 1152 total runs


In [23]:
def build_and_run_qaoa(node_ids, depot_id, dist_map, global_max, eff_k,
                       reps, penalty_mult, optimizer_name, maxiter, n_restarts, seed):
    all_ids = [depot_id] + list(node_ids)
    m1 = len(all_ids); N = len(node_ids)
    dist_mat = np.zeros((m1, m1))
    for qi, a in enumerate(all_ids):
        for qj, b in enumerate(all_ids):
            if a != b and (a, b) in dist_map: dist_mat[qi, qj] = dist_map[(a, b)]
    gmax = global_max if global_max > 0 else max(float(np.max(dist_mat)), 1.0)
    dist_norm = dist_mat / gmax
    penalty = penalty_mult * N
    leaf_seed = (seed + int(hashlib.md5(str(sorted(node_ids)).encode()).hexdigest(), 16)) % (2**31)
    ag.random_seed = leaf_seed; np.random.seed(leaf_seed)

    if eff_k == 1:
        formulation = "pos-idx"
        qp = QuadraticProgram()
        for ci in range(N):
            for ti in range(N): qp.binary_var(f"p_{ci}_{ti}")
        lin, quad = {}, {}
        for ci in range(N):
            di = ci + 1
            lin[f"p_{ci}_0"] = lin.get(f"p_{ci}_0", 0) + float(dist_norm[0, di])
            lin[f"p_{ci}_{N-1}"] = lin.get(f"p_{ci}_{N-1}", 0) + float(dist_norm[di, 0])
        for ti in range(N - 1):
            for ci in range(N):
                di = ci + 1
                for cj in range(N):
                    dj = cj + 1
                    if ci != cj:
                        key = (f"p_{ci}_{ti}", f"p_{cj}_{ti+1}")
                        quad[key] = quad.get(key, 0) + float(dist_norm[di, dj])
        qp.minimize(linear=lin, quadratic=quad)
        for ci in range(N):
            qp.linear_constraint({f"p_{ci}_{ti}": 1 for ti in range(N)}, sense="==", rhs=1, name=f"cust_{ci}")
        for ti in range(N):
            qp.linear_constraint({f"p_{ci}_{ti}": 1 for ci in range(N)}, sense="==", rhs=1, name=f"pos_{ti}")
    else:
        formulation = "edge"
        qp = QuadraticProgram()
        for ei in range(m1):
            for ej in range(m1):
                if ei != ej: qp.binary_var(f"x_{ei}_{ej}")
        qp.minimize(linear={f"x_{ei}_{ej}": float(dist_norm[ei, ej])
                     for ei in range(m1) for ej in range(m1) if ei != ej})
        for ei in range(1, m1):
            qp.linear_constraint({f"x_{ei}_{ej}": 1 for ej in range(m1) if ej != ei}, sense="==", rhs=1, name=f"out_{ei}")
            qp.linear_constraint({f"x_{ej}_{ei}": 1 for ej in range(m1) if ej != ei}, sense="==", rhs=1, name=f"in_{ei}")
        qp.linear_constraint({f"x_0_{ej}": 1 for ej in range(1, m1)}, sense="==", rhs=eff_k, name="out_0")
        qp.linear_constraint({f"x_{ej}_0": 1 for ej in range(1, m1)}, sense="==", rhs=eff_k, name="in_0")

    qubo = QuadraticProgramToQubo(penalty=penalty).convert(qp)
    n_qubits = len(qubo.variables)
    var_names = [v.name for v in qubo.variables]
    ising_op, offset = qubo.to_ising()
    raw = np.array([abs(c) for _, c in ising_op.to_list()], dtype=float)
    op_scale = float(np.max(raw)) if len(raw) > 0 and np.max(raw) > 0 else 1.0
    ising_n = ising_op / op_scale

    backend = AerSimulator(method="statevector", seed_simulator=leaf_seed)
    est = Estimator(run_options={"seed_simulator": leaf_seed})
    ansatz = QAOAAnsatz(ising_n, reps=reps)
    tqc = transpile(ansatz, backend=backend, optimization_level=3)
    def energy_fn(theta, _tqc=tqc):
        theta = np.asarray(theta, dtype=float).ravel()
        return float(est.run([_tqc], [ising_n], parameter_values=[theta]).result().values[0]) * op_scale + offset
    if optimizer_name == "ADAM":     opt = ADAM(maxiter=maxiter, amsgrad=False)
    elif optimizer_name == "COBYLA": opt = COBYLA(maxiter=maxiter)
    elif optimizer_name == "SPSA":   opt = SPSA(maxiter=maxiter)
    else: raise ValueError(optimizer_name)

    t0 = time.time()
    best_res = None
    for _ in range(n_restarts):
        x0 = 2 * np.pi * np.random.rand(ansatz.num_parameters)
        res = opt.minimize(fun=energy_fn, x0=x0)
        if best_res is None or res.fun < best_res.fun: best_res = res
    circ = tqc.copy()
    if not circ.cregs: circ.measure_all()
    bound = circ.assign_parameters(best_res.x, inplace=False)
    counts = backend.run(bound, shots=SHOTS).result().get_counts()
    wall_time = time.time() - t0

    # -- Decode --
    total = int(sum(counts.values()))
    total_q = max(len(s.replace(" ", "")) for s in counts)
    valid_costs = {}; valid_route_costs = {}

    if eff_k == 1:
        pvm = {}
        for vi, vn in enumerate(var_names):
            if vn.startswith("p_"):
                pts = vn.split("_"); pvm[vi] = (int(pts[1]), int(pts[2]))
        for bitstr, cnt in counts.items():
            s = bitstr.replace(" ", "")
            bf = np.array([1 if c == "1" else 0 for c in s], dtype=np.int8)
            if len(bf) < total_q: bf = np.concatenate([np.zeros(total_q - len(bf), dtype=np.int8), bf])
            bf = bf[::-1]
            p = np.zeros((N, N), dtype=int)
            for vi, (ci, ti) in pvm.items():
                if vi < len(bf): p[ci, ti] = bf[vi]
            if not (np.all(p.sum(axis=1) == 1) and np.all(p.sum(axis=0) == 1)): continue
            perm = tuple(int(np.argmax(p[:, ti])) for ti in range(N))
            rids = [all_ids[pi + 1] for pi in perm]
            cost = route_distance(rids, depot_id, dist_map)
            lbl = str(perm)
            valid_costs[lbl] = valid_costs.get(lbl, 0) + int(cnt)
            valid_route_costs[lbl] = cost
    else:
        ak, ai_a, aj_a = [], [], []
        for ki_idx, vn in enumerate(var_names):
            if vn.startswith("x_"):
                _, si, sj = vn.split("_"); ak.append(ki_idx); ai_a.append(int(si)); aj_a.append(int(sj))
        arc_k = np.array(ak); arc_i = np.array(ai_a); arc_j = np.array(aj_a)
        for bitstr, cnt in counts.items():
            s = bitstr.replace(" ", "")
            bf = np.array([1 if c == "1" else 0 for c in s], dtype=np.int8)
            if len(bf) < total_q: bf = np.concatenate([np.zeros(total_q - len(bf), dtype=np.int8), bf])
            bf = bf[::-1]
            b = bf[arc_k]; idx = np.flatnonzero(b)
            od = np.bincount(arc_i[idx], minlength=m1); id_ = np.bincount(arc_j[idx], minlength=m1)
            if not (od[0] == eff_k and id_[0] == eff_k and np.all(od[1:] == 1) and np.all(id_[1:] == 1)): continue
            edges = list(zip(arc_i[idx].tolist(), arc_j[idx].tolist()))
            succ = defaultdict(list)
            for ea, eb in edges: succ[ea].append(eb)
            depot_exits = succ.get(0, [])
            if len(depot_exits) != eff_k: continue
            bad = False; visited_g = set(); completed = 0
            for sn in depot_exits:
                cur, path = sn, set()
                while True:
                    if cur == 0: completed += 1; break
                    if cur in path or cur in visited_g: bad = True; break
                    path.add(cur)
                    nxt = succ.get(cur, [])
                    if len(nxt) != 1: bad = True; break
                    cur = nxt[0]
                if bad: break
                visited_g |= path
            if bad or completed != eff_k or visited_g != set(range(1, m1)): continue
            succ_nd = {ea: eb for ea, eb in edges if ea != 0}
            starts = [eb for ea, eb in edges if ea == 0]
            total_cost = 0.0
            for sn in starts:
                r = []; cur = sn
                while cur != 0: r.append(cur); cur = succ_nd.get(cur, 0)
                total_cost += route_distance([all_ids[qi] for qi in r], depot_id, dist_map)
            lbl = "".join("1" if x else "0" for x in b)
            valid_costs[lbl] = valid_costs.get(lbl, 0) + int(cnt)
            valid_route_costs[lbl] = total_cost

    valid_frac = int(sum(valid_costs.values())) / total if total > 0 else 0.0
    n_valid_unique = len(valid_costs)
    if not valid_costs:
        return {"n_qubits": n_qubits, "formulation": formulation, "final_energy": round(best_res.fun, 6),
                "wall_time_s": round(wall_time, 2), "topk_cost": float("inf"), "mp_cost": float("inf"),
                "global_cost": float("inf"), "valid_frac": 0.0, "n_valid_unique": 0,
                "zero_valid": True, "best_prob_rank": -1}
    sbp = sorted(valid_costs, key=valid_costs.__getitem__, reverse=True)
    mp_cost = valid_route_costs[sbp[0]]
    topk_lbls = sbp[:DECODE_K]
    topk_cost = min(valid_route_costs[l] for l in topk_lbls)
    gl = min(valid_costs.keys(), key=lambda l: valid_route_costs[l])
    global_cost = valid_route_costs[gl]
    best_prob_rank = sbp.index(gl) + 1
    return {"n_qubits": n_qubits, "formulation": formulation, "final_energy": round(best_res.fun, 6),
            "wall_time_s": round(wall_time, 2), "topk_cost": round(topk_cost, 4),
            "mp_cost": round(mp_cost, 4), "global_cost": round(global_cost, 4),
            "valid_frac": round(valid_frac, 6), "n_valid_unique": n_valid_unique,
            "zero_valid": False, "best_prob_rank": best_prob_rank}

print("QAOA runner ready.")


QAOA runner ready.


In [24]:
results = []; run_id = 0
print("Pre-computing classical optima ...")
OPTIMA = {}
for inst_name, idata in INSTANCES.items():
    nids = [n.id for n in idata["nodes"]]
    for k in SWEEP_K:
        ek = min(k, len(nids))
        opt = classical_optimal(nids, ek, idata["depot"].id, idata["dist_map"])
        OPTIMA[(inst_name, ek)] = opt
        print(f"  {inst_name} k={ek}: optimal = {opt:.2f}")

total_runs = (len(INSTANCES) * len(SWEEP_K) * len(SWEEP_REPS) * len(SWEEP_PENALTY)
              * len(SWEEP_OPTIMIZER) * len(SWEEP_MAXITER) * len(SWEEP_RESTARTS))
print(f"\nStarting {total_runs} runs ...\n")

for inst_name, idata in INSTANCES.items():
    nids = [n.id for n in idata["nodes"]]
    for k in SWEEP_K:
        ek = min(k, len(nids))
        opt_cost = OPTIMA[(inst_name, ek)]
        for reps in SWEEP_REPS:
            for pen in SWEEP_PENALTY:
                for oname in SWEEP_OPTIMIZER:
                    for mi in SWEEP_MAXITER:
                        for rst in SWEEP_RESTARTS:
                            run_id += 1
                            tag = f"[{run_id}/{total_runs}] {inst_name} k={ek} p={reps} pen={pen} {oname} mi={mi} r={rst}"
                            try:
                                res = build_and_run_qaoa(nids, idata["depot"].id, idata["dist_map"],
                                                         idata["global_max"], ek, reps, pen, oname, mi, rst, SEED)
                                def _gap(c):
                                    return round((c - opt_cost) / opt_cost * 100, 4) if opt_cost > 0 and c < float("inf") else None
                                row = {"run_id": run_id, "instance": inst_name, "k": ek,
                                       "reps": reps, "penalty_mult": pen, "optimizer": oname,
                                       "maxiter": mi, "restarts": rst, "n_qubits": res["n_qubits"],
                                       "formulation": res["formulation"], "final_energy": res["final_energy"],
                                       "topk_cost": res["topk_cost"], "mp_cost": res["mp_cost"],
                                       "global_cost": res["global_cost"], "optimal_cost": round(opt_cost, 4),
                                       "topk_gap_pct": _gap(res["topk_cost"]), "mp_gap_pct": _gap(res["mp_cost"]),
                                       "global_gap_pct": _gap(res["global_cost"]), "valid_frac": res["valid_frac"],
                                       "n_valid_unique": res["n_valid_unique"], "zero_valid": res["zero_valid"],
                                       "best_prob_rank": res["best_prob_rank"], "wall_time_s": res["wall_time_s"]}
                                results.append(row)
                                g = f"gap={row['topk_gap_pct']:.2f}%" if row["topk_gap_pct"] is not None else "NO VALID"
                                print(f"  {tag}  {g}  vf={res['valid_frac']:.3%}  {res['wall_time_s']:.1f}s")
                            except Exception as e:
                                print(f"  {tag}  ERROR: {e}")
                                results.append({"run_id": run_id, "instance": inst_name, "k": ek,
                                                "reps": reps, "penalty_mult": pen, "optimizer": oname,
                                                "maxiter": mi, "restarts": rst, "error": str(e)})

print(f"\nSweep complete: {len(results)} results.")


Pre-computing classical optima ...
  RioClaroPostToy_5_0 k=1: optimal = 12214.56
  RioClaroPostToy_5_0 k=2: optimal = 15143.35
  RioClaroPostToy_5_0 k=3: optimal = 18342.58
  RioClaroPostToy_5_0 k=4: optimal = 21555.96
  RioClaroPostToy_5_1 k=1: optimal = 17499.97
  RioClaroPostToy_5_1 k=2: optimal = 19642.36
  RioClaroPostToy_5_1 k=3: optimal = 23577.94
  RioClaroPostToy_5_1 k=4: optimal = 29375.85
  RioClaroPostToy_5_2 k=1: optimal = 10216.15
  RioClaroPostToy_5_2 k=2: optimal = 13612.99
  RioClaroPostToy_5_2 k=3: optimal = 17329.11
  RioClaroPostToy_5_2 k=4: optimal = 22134.28

Starting 1152 runs ...

  [1/1152] RioClaroPostToy_5_0 k=1 p=1 pen=2.0 COBYLA mi=50 r=1  NO VALID  vf=0.000%  4.8s
  [2/1152] RioClaroPostToy_5_0 k=1 p=1 pen=2.0 COBYLA mi=50 r=3  gap=0.00%  vf=0.130%  14.3s
  [3/1152] RioClaroPostToy_5_0 k=1 p=1 pen=2.0 COBYLA mi=200 r=1  NO VALID  vf=0.000%  4.4s
  [4/1152] RioClaroPostToy_5_0 k=1 p=1 pen=2.0 COBYLA mi=200 r=3  gap=0.00%  vf=0.130%  14.0s
  [5/1152] RioClar

In [25]:
CSV_PATH = "qaoa_sweep_results.csv"
fieldnames = ["run_id", "instance", "k", "reps", "penalty_mult", "optimizer",
              "maxiter", "restarts", "n_qubits", "formulation", "final_energy",
              "topk_cost", "mp_cost", "global_cost", "optimal_cost",
              "topk_gap_pct", "mp_gap_pct", "global_gap_pct",
              "valid_frac", "n_valid_unique", "zero_valid", "best_prob_rank", "wall_time_s"]
with open(CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    for row in results: writer.writerow(row)
print(f"Exported {len(results)} rows to {CSV_PATH}")
valid_runs = [r for r in results if r.get("topk_gap_pct") is not None]
if valid_runs:
    optimal = [r for r in valid_runs if abs(r["topk_gap_pct"]) < 0.01]
    print(f"Valid: {len(valid_runs)}/{len(results)}  Optimal: {len(optimal)}/{len(valid_runs)} ({len(optimal)/len(valid_runs)*100:.1f}%)")
    best = min(valid_runs, key=lambda r: r["topk_gap_pct"])
    print(f"Best: {best['instance']} k={best['k']} p={best['reps']} pen={best['penalty_mult']} "
          f"{best['optimizer']} mi={best['maxiter']} r={best['restarts']} gap={best['topk_gap_pct']:.4f}%")


Exported 1152 rows to qaoa_sweep_results.csv
Valid: 884/1152  Optimal: 596/884 (67.4%)
Best: RioClaroPostToy_5_0 k=1 p=1 pen=2.0 COBYLA mi=50 r=3 gap=0.0000%


In [26]:
valid_runs = [r for r in results if r.get("topk_gap_pct") is not None]
if not valid_runs:
    print("No valid solutions found.")
else:
    for title, key, vals in [
        ("OPTIMIZER", "optimizer", SWEEP_OPTIMIZER),
        ("CIRCUIT DEPTH (reps)", "reps", SWEEP_REPS),
        ("PENALTY MULTIPLIER", "penalty_mult", SWEEP_PENALTY),
        ("FLEET SIZE (k)", "k", SWEEP_K),
        ("INSTANCE", "instance", list(INSTANCES.keys())),
    ]:
        print(f"\n{'='*70}")
        print(f"OPTIMALITY RATE BY {title}")
        print(f"{'='*70}")
        for val in vals:
            sub = [r for r in valid_runs if r[key] == val]
            if not sub: continue
            n_opt = sum(1 for r in sub if abs(r["topk_gap_pct"]) < 0.01)
            avg_gap = np.mean([r["topk_gap_pct"] for r in sub])
            avg_vf = np.mean([r["valid_frac"] for r in sub])
            print(f"  {str(val):>12s}:  {n_opt:>4d}/{len(sub):<4d} optimal "
                  f"({n_opt/len(sub)*100:5.1f}%)  avg_gap={avg_gap:7.3f}%  avg_valid={avg_vf:.4%}")

    print(f"\n{'='*70}")
    print("TOP 20 CONFIGURATIONS (by topk_gap, then valid_frac)")
    print(f"{'='*70}")
    ranked = sorted(valid_runs, key=lambda r: (r["topk_gap_pct"], -r["valid_frac"]))
    print(f"  {'#':>3s}  {'Instance':>22s}  {'k':>2s}  {'p':>2s}  {'Pen':>5s}  "
          f"{'Opt':>6s}  {'MI':>4s}  {'R':>2s}  {'Gap%':>8s}  {'Valid%':>8s}  "
          f"{'Rank':>4s}  {'Time':>6s}")
    for idx, r in enumerate(ranked[:20], 1):
        print(f"  {idx:>3d}  {r['instance']:>22s}  {r['k']:>2d}  {r['reps']:>2d}  "
              f"{r['penalty_mult']:>5.1f}  {r['optimizer']:>6s}  {r['maxiter']:>4d}  "
              f"{r['restarts']:>2d}  {r['topk_gap_pct']:>7.3f}%  "
              f"{r['valid_frac']:>7.3%}  {r['best_prob_rank']:>4d}  "
              f"{r['wall_time_s']:>5.1f}s")



OPTIMALITY RATE BY OPTIMIZER
        COBYLA:   410/540  optimal ( 75.9%)  avg_gap=  1.923%  avg_valid=3.8011%
          ADAM:   186/344  optimal ( 54.1%)  avg_gap=  5.205%  avg_valid=0.1544%

OPTIMALITY RATE BY CIRCUIT DEPTH (reps)
             1:   134/198  optimal ( 67.7%)  avg_gap=  3.021%  avg_valid=0.8337%
             2:   178/254  optimal ( 70.1%)  avg_gap=  2.641%  avg_valid=0.9296%
             3:   136/204  optimal ( 66.7%)  avg_gap=  3.938%  avg_valid=1.8577%
             5:   148/228  optimal ( 64.9%)  avg_gap=  3.317%  avg_valid=5.8138%

OPTIMALITY RATE BY PENALTY MULTIPLIER
           2.0:   208/303  optimal ( 68.6%)  avg_gap=  3.105%  avg_valid=2.0403%
           5.0:   190/287  optimal ( 66.2%)  avg_gap=  3.392%  avg_valid=2.6230%
          10.0:   198/294  optimal ( 67.3%)  avg_gap=  3.110%  avg_valid=2.4990%

OPTIMALITY RATE BY FLEET SIZE (k)
             1:   175/264  optimal ( 66.3%)  avg_gap=  3.448%  avg_valid=2.4605%
             2:    75/211  optimal ( 35.5%)  